# Job Salary Prediction

**Tabular Regression Project** — Predict job salaries from job listing features.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Data Science Job Salaries (Kaggle)
Target: `salary_in_usd`

## 2 · Project Overview

We predict **data science job salaries** using job listing features including experience level, employment type, job title, company size, and location. This is a practical regression problem for career planning and compensation benchmarking.

## 3 · Learning Objectives

1. Work with job market/salary datasets.
2. Handle high-cardinality categorical features (job titles).
3. Apply regression models to compensation prediction.
4. Use LazyPredict and FLAML for model selection.
5. Understand salary prediction limitations.

## 4 · Problem Statement

Predict the **salary in USD** for data science positions based on experience level, employment type, job title, company size, and location.

## 5 · Why This Project Matters

- **Salary transparency** helps job seekers negotiate better.
- Companies use salary prediction for competitive compensation planning.
- Experience level, location, and role type are key drivers worth understanding.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: ruchi798/data-science-job-salaries |
| **Target** | salary_in_usd |
| **Features** | experience_level, employment_type, job_title, company_size, company_location, employee_residence |

## 7 · Dataset Source and License

- **Source**: AI-Jobs.net salary survey data, available on Kaggle.
- **License**: CC0 Public Domain.
- **Note**: Self-reported salaries with potential biases.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'salary_in_usd'
np.random.seed(SEED)

## 11 · Dataset Download

In [ ]:
import kagglehub, glob
try:
    path = kagglehub.dataset_download('ruchi798/data-science-job-salaries')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')

# Ensure target exists
if TARGET not in df.columns:
    sal_cols = [c for c in df.columns if 'salary' in c.lower()]
    TARGET_ACTUAL = sal_cols[0] if sal_cols else df.select_dtypes('number').columns[-1]
    print(f'Using target: {TARGET_ACTUAL}')
else:
    TARGET_ACTUAL = TARGET
print(f'Target range: [{df[TARGET_ACTUAL].min():,.0f}, {df[TARGET_ACTUAL].max():,.0f}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[TARGET_ACTUAL].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Salary Distribution')
if 'experience_level' in df.columns:
    df.groupby('experience_level')[TARGET_ACTUAL].mean().sort_values().plot.barh(ax=axes[0,1])
    axes[0,1].set_title('Avg Salary by Experience')
if 'company_size' in df.columns:
    df.groupby('company_size')[TARGET_ACTUAL].mean().sort_values().plot.barh(ax=axes[1,0])
    axes[1,0].set_title('Avg Salary by Company Size')
if 'employment_type' in df.columns:
    df.groupby('employment_type')[TARGET_ACTUAL].mean().sort_values().plot.barh(ax=axes[1,1])
    axes[1,1].set_title('Avg Salary by Employment Type')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET_ACTUAL].describe())
print(f'\nSkewness: {df[TARGET_ACTUAL].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Drop unnamed index and redundant salary columns
for col in df.columns:
    if 'unnamed' in col.lower(): df = df.drop(columns=[col])
    elif col != TARGET_ACTUAL and 'salary' in col.lower(): df = df.drop(columns=[col])
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100: df = df.drop(columns=[col])
    else: df[col] = LabelEncoder().fit_transform(df[col].fillna('Unknown'))
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET_ACTUAL])
y = df[TARGET_ACTUAL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:,.0f}, MAE={baseline_mae:,.0f}, R2={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R2: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R2={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation

- **Experience level** is the strongest salary predictor (Senior/Executive >> Entry).
- **Company size** and **location** matter — large US companies pay significantly more.
- **Job title** adds specificity but can overfit with many categories.
- These insights help both job seekers and HR teams benchmark compensation.

## 26 · Limitations

- Self-reported data with potential biases.
- Data science roles only — not generalizable.
- No benefits/equity information.
- Salary ranges vary significantly within categories.

## 27 · How to Improve

1. Add skills/tech stack features.
2. Include industry sector.
3. Add years of experience (continuous).
4. Include education level.
5. Target encoding for job titles.

## 28 · Production Considerations

- Regular updates as market conditions change.
- Pay equity compliance.
- Fairness auditing across demographics.
- Present ranges, not point estimates.

## 29 · Common Mistakes

1. Including redundant salary columns as features.
2. Using raw job titles with too many categories.
3. Not accounting for currency differences.
4. Ignoring self-reporting bias.

## 30 · Mini Challenge

1. Predict log(salary) and compare.
2. Target-encode job titles.
3. Build separate models by experience level.
4. Add a 'remote_ratio' interaction feature.

## 31 · Final Summary

- Salary prediction is practical for career planning and HR.
- Experience level and location dominate compensation.
- Self-reported data introduces noise.
- Boosting models handle the categorical features well.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')